# 21 · Set 6: the next-experiments plan (4_2_Next_Routing_Experiments_Plan.docx) on the 400 development questions (A100)

**The questions:** when the retrieval looks incomplete, is it better to retry it than to escalate to Qwen 32B? Does a short, focused
context after a retry help more than the full union of passages? Can Llama 8B repair its own answer cheaply once new evidence arrives?

| Plan item | What this notebook does | Primary variant (named before running) |
|---|---|---|
| 1 Different remedies | Every question gets every action: keep the Llama 8B answer; Qwen 32B on the same passages; retry retrieval and answer again. A three-action rule (retriever probe, then answerer probe) is compared with the answerer-only gate, the retriever-only gate, random with the same action mix, and the cheap fixed setup | Retry if the retriever probe is low; else escalate if the answer probe is low; else keep. Two thresholds fitted on training questions under the fixed setup's compute |
| 2 Adaptive context | After a retry: the full union of passages vs a bounded shortlist of 2, 3 or 4 (relevance + coverage − redundancy, fixed weights). Gold-only context as a diagnostic ceiling | Probe-triggered retry, then the bounded shortlist with k chosen on training questions |
| 3 Answer revision | After a retry: Llama 8B answers from scratch, Llama 8B revises its first answer, Qwen 32B answers | One Llama 8B revision, no extra escalation |
| 6, 5, 4 (CPU) | Expert ablation, stable escalation benefit, cheap support features (same code as the CPU run already reported) | As in the plan |

What runs, in order (every stage is resumable: if the session stops, run the notebook again and it continues):

| Stage | Model | Answers | My estimate |
|---|---|---|---|
| Install vLLM, copy the two models | - | - | about 8–12 min |
| 1. Llama 8B: first answer, answers on every new context, revisions | Llama 8B, greedy, up to 512 tokens | about 2,400 | about 10–20 min |
| 2. Qwen 32B (AWQ): first shortlist and every new context | Qwen 32B, greedy, up to 512 tokens | about 1,500 | about 15–30 min |
| 3. Analysis (items 1–3) and the CPU items (6, 5, 4) | none | - | about 10–15 min |

These are estimates, not measurements; notebook 19 took more than twice my estimate, so allow up to 1.5 hours. No new retrieval is run: the
three retrieval tries saved by notebooks 11 and 17 are reused. Nothing is frozen and nothing touches new questions: everything here is development.

Output: `runs/Set_2__question-reading-routers/<timestamp>_<GPU>_set6/` with `results_actions/` (summary.md, set6_actions.xlsx, charts) and `results_cpu/`.

**Runtime → Change runtime type → A100 GPU.**

In [ ]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
EXPERIMENT  = "Set_2__question-reading-routers"
OLD_SWEEP_EXPERIMENT = "Set_1__3-modality-size"
FRESH_DATA  = f"{PROJECT_DIR}/data/hotpotqa_q/fresh200_c20_s43"
TIME_LIMIT_MIN = 100                  # stops cleanly at this limit; run the notebook again to continue
CV_REPEATS = 2                        # repeats of the 5-fold cross-validation over all 400 questions
NEW_RUN = False                       # False = continue the latest run of this notebook
DRY_RUN = False                       # True = fake models on CPU, tests the whole notebook in a few minutes

EXPECTED_CODE = {
 "ssw.py": "c9fbdc7f6b91c58b607021b8be8c03ae2ff67dc1024fd2e24fffeb18e239ed98",
 "sweep.py": "9382292c6fe396cf79c73b212aaf6c2d677cae7f9311c3a5304f0578b52e6bb6",
 "routers.py": "8a293df1c8955716950f535cf9a3ced24439e00250173da1a3df110c0ed87304",
 "hidden.py": "574e6c1390e47c6de833a0c10b32c24eed0dee73a35eabc17f58842835e7f105",
 "final.py": "f18a6fba3d1bdd1f6700f6dc535f10ec0c41dbf5192b7b39ad097f8a1b88be53",
 "retriever_check.py": "fde08c549ecbd6bfc5527a42c731aa51dcff687a64818d68cf9cb30090e017e3",
 "ensemble.py": "5c5e87175d2484d806fda913a9555f71378a6031f050c37e89c20305942b76fc",
 "experts.py": "17ed39957e54853d178b47fa73ba328b50f0589b129ccb191a2efb0a2dea466b",
 "next42.py": "341a5eeff3fffb477bb3a21c00cbb53814bbc44d13303bebe3d81643f02b05a7"
}

In [ ]:
import subprocess, sys, os, json, shutil, time
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if not DRY_RUN:
    assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU (or set DRY_RUN = True)."
    if "T4" in gpu:
        raise SystemExit("This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.\n"
                         "Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.")
    if "A100" not in gpu and "G4" not in gpu and "RTX PRO 6000" not in gpu:
        print("Note: planned for an A100 or G4; this GPU works but timings will differ.")
GPU_TAG = "cpu" if not gpu else gpu.split(",")[0].replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].split("-")[0]
T_START = time.time()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Check that Drive is serving the current code; find the earlier runs

In [ ]:
import glob, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
CODE_DIR = "/content/set6code"
os.makedirs(CODE_DIR, exist_ok=True)
stale = []
for f, want in EXPECTED_CODE.items():
    shutil.copy(f"{PROJECT_DIR}/code/{f}", f"{CODE_DIR}/{f}")
    if sha(f"{CODE_DIR}/{f}") != want:
        stale.append(f)
assert not stale, (f"Drive is serving a different version of {stale} than this notebook was made for. Wait two minutes, then "
                   "Runtime -> Disconnect and delete runtime, and run again.")
sweeps = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{OLD_SWEEP_EXPERIMENT}/*")) if "_dryrun" not in d and os.path.exists(f"{d}/results/per_question.jsonl")
          and os.path.exists(f"{d}/sweep_status.json") and json.load(open(f"{d}/sweep_status.json")).get("complete")]
hidden = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*_hidden")) if os.path.exists(f"{d}/step/hidden_retrieve_q7.jsonl")
          and os.path.exists(f"{d}/step/hidden_answer_l8.jsonl")]
fresh = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*_final")) if os.path.exists(f"{d}/step/hidden_retrieve_q7.jsonl")
         and os.path.exists(f"{d}/step/hidden_answer_l8.jsonl")]
assert sweeps and hidden and fresh, "Needs the finished runs of notebooks 11, 16, 17 and 18."
OLD_SWEEP, OLD_HIDDEN, FRESH_RUN = sweeps[-1], hidden[-1], fresh[-1]
OLD_DATA = json.load(open(f"{OLD_SWEEP}/config.json"))["data_dir"]
SESSION = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d-%H%M%S")
tag = "_set6_dryrun" if DRY_RUN else "_set6"
prev = sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*{tag}"))
RUN_DIR = prev[-1] if (prev and not NEW_RUN) else f"{PROJECT_DIR}/runs/{EXPERIMENT}/{SESSION}_{GPU_TAG}{tag}"
SESSION_DIR = f"{RUN_DIR}/sessions/{SESSION}"
for d in (f"{RUN_DIR}/code", f"{RUN_DIR}/logs", f"{RUN_DIR}/step", SESSION_DIR):
    os.makedirs(d, exist_ok=True)
for f in EXPECTED_CODE:
    shutil.copy(f"{CODE_DIR}/{f}", f"{RUN_DIR}/code/{f}")
TIMES = {}
print("Code files: current.")
print("Old questions:  ", OLD_SWEEP, "\n                ", OLD_HIDDEN)
print("Fresh questions:", FRESH_RUN)
print("Run folder:     ", RUN_DIR)

In [ ]:
MODELS = {   # key: model, folder under models/, parameters (for compute), nominal size (for the AgentTTS budget formula)
    "q3":  {"id": "Qwen/Qwen2.5-3B-Instruct",        "dir": "Qwen2.5-3B-Instruct",        "params": 3.09e9,  "nominal": 3e9},
    "q7":  {"id": "Qwen/Qwen2.5-7B-Instruct",        "dir": "Qwen2.5-7B-Instruct",        "params": 7.62e9,  "nominal": 7e9},
    "q32": {"id": "Qwen/Qwen2.5-32B-Instruct-AWQ",   "dir": "Qwen2.5-32B-Instruct-AWQ",   "params": 32.76e9, "nominal": 32e9},
    "l3":  {"id": "meta-llama/Llama-3.2-3B-Instruct", "dir": "Llama-3.2-3B-Instruct",     "params": 3.21e9,  "nominal": 3e9},
    "l8":  {"id": "meta-llama/Llama-3.1-8B-Instruct", "dir": "Llama-3.1-8B-Instruct",     "params": 8.03e9,  "nominal": 8e9},
}
LOCAL = "/content/models"
for m in MODELS.values():
    m["path"] = f"{LOCAL}/{m['dir']}"

MODELS = {k: v for k, v in MODELS.items() if k in ("q7", "l8", "q32")}
json.dump(MODELS, open(f"{RUN_DIR}/models.json", "w"), indent=1)
json.dump({"experiment": EXPERIMENT, "set": "6 next-experiments plan", "old_sweep": OLD_SWEEP, "old_hidden": OLD_HIDDEN, "fresh_run": FRESH_RUN,
           "old_data": OLD_DATA, "fresh_data": FRESH_DATA, "cv_repeats": CV_REPEATS, "gpu": gpu, "dry_run": DRY_RUN, "code": EXPECTED_CODE, "started": SESSION},
          open(f"{RUN_DIR}/config.json", "w"), indent=1)

def minutes_left():
    return max(0.0, TIME_LIMIT_MIN - (time.time() - T_START) / 60)

def run_stage(args, log_name):
    cmd = [sys.executable, "-u", f"{CODE_DIR}/next42.py"] + [str(a) for a in args]
    t0 = time.time()
    with open(f"{RUN_DIR}/logs/{log_name}", "a") as log, open(f"{SESSION_DIR}/{log_name}", "a") as slog:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); slog.write(line)
            if not any(k in line for k in ("it/s]", "s/it]", "Loading checkpoint", "INFO ", "(EngineCore", "WARNING ", "Warning", "warn")):
                print(line, end="")
        p.wait()
    TIMES[log_name.replace(".log", "")] = TIMES.get(log_name.replace(".log", ""), 0) + round(time.time() - t0)
    json.dump(TIMES, open(f"{SESSION_DIR}/stage_times.json", "w"), indent=1)
    print(f"--- {log_name}: exit {p.returncode}, {time.time() - t0:.0f}s, {minutes_left():.0f} min left of the limit")
    assert p.returncode == 0, f"stage failed - see {RUN_DIR}/logs/{log_name}"

GEN = ["--old_sweep", OLD_SWEEP, "--old_data", OLD_DATA, "--fresh_run", FRESH_RUN, "--fresh_data", FRESH_DATA, "--run_dir", RUN_DIR,
       "--models_json", f"{RUN_DIR}/models.json"]
def pending(key):
    r = subprocess.run([sys.executable, f"{CODE_DIR}/next42.py", "generate", "--model", key, "--count_only"] + GEN, capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-3000:]
    return json.loads(r.stdout.strip().splitlines()[-1])["pending"]
TODO = {k: pending(k) for k in ("l8", "q32")}
print("answers still to generate:", TODO)

### Copy the models in the background, install vLLM

In [ ]:
import threading
COPY = {"err": None, "done": set(), "seconds": {}}
def _copy_all(keys):
    try:
        for key in keys:
            m = MODELS[key]
            src = f"{PROJECT_DIR}/models/{m['dir']}"
            if not os.path.exists(f"{m['path']}/config.json"):
                assert os.path.exists(f"{src}/config.json"), f"{src} missing - run notebook 10 first."
                t = time.time()
                shutil.rmtree(m["path"] + ".part", ignore_errors=True)
                shutil.copytree(src, m["path"] + ".part")
                os.rename(m["path"] + ".part", m["path"])               # the folder appears only when the copy is complete
                COPY["seconds"][key] = round(time.time() - t)
            COPY["done"].add(key)
    except Exception as e:
        COPY["err"] = e

def wait_local(key):
    if DRY_RUN:
        return
    t = time.time()
    while key not in COPY["done"]:
        if COPY["err"]:
            raise COPY["err"]
        time.sleep(5)
    if time.time() - t > 10:
        print(f"waited {time.time() - t:.0f}s for the copy of {MODELS[key]['dir']}")
    TIMES[f"copy_{key}"] = COPY["seconds"].get(key, 0)

todo_models = [k for k in ("l8", "q32") if TODO[k]]
if not DRY_RUN and todo_models:
    os.makedirs(LOCAL, exist_ok=True)
    threading.Thread(target=_copy_all, args=(todo_models,), daemon=True).start()
    print("copying in the background:", [MODELS[k]["dir"] for k in todo_models])
if not DRY_RUN and todo_models:
    t = time.time()
    !pip install -q vllm
    !pip uninstall -y -q torchaudio   # Colab's torchaudio is built for another CUDA version and breaks imports
    chk = subprocess.run([sys.executable, "-c", "from vllm import LLM; import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"],
                         capture_output=True, text=True)
    print(chk.stdout or chk.stderr[-2000:])
    assert chk.returncode == 0, "vLLM does not import - Runtime -> Disconnect and delete runtime, then run again."
    TIMES["install_vllm"] = round(time.time() - t)
else:
    print("nothing to generate - no install needed")
ver = subprocess.run([sys.executable, "-c", "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"],
                     capture_output=True, text=True).stdout.strip()
print(ver)

### Generate the answers (Llama 8B, then Qwen 32B; resumable)

In [ ]:
BACKEND = "mock" if DRY_RUN else "vllm"
for key in ("l8", "q32"):
    if not pending(key):
        print(f"{MODELS[key]['dir']}: every answer is already there")
        continue
    if minutes_left() < 8:
        print(f"Time limit close ({minutes_left():.0f} min left) - {key} not started. Run this notebook again to continue.")
        break
    wait_local(key)
    run_stage(["generate", "--model", key, "--backend", BACKEND, "--time_limit_min", max(1.0, minutes_left() - 4)] + GEN, f"generate_{key}.log")
    if pending(key):
        print(f"{key} is not complete yet. Run this notebook again to continue.")
        break
ALL_DONE = not pending("l8") and not pending("q32")
print("\nAll answers generated." if ALL_DONE else "\nNOT COMPLETE YET - run this notebook again to continue.")

### Analysis: items 1–3 (different remedies, context size, revision), then the CPU items 6, 5 and 4

In [ ]:
if ALL_DONE:
    from IPython.display import Image, Markdown, display
    SETS = ["--old_sweep", OLD_SWEEP, "--old_hidden", OLD_HIDDEN, "--old_data", OLD_DATA, "--fresh_run", FRESH_RUN, "--fresh_data", FRESH_DATA, "--run_dir", RUN_DIR]
    for flag, sub in (([], "results_actions"), (["--first_from_extraction"], "results_actions_probe_answer")):
        if not os.path.exists(f"{RUN_DIR}/{sub}/summary.md"):
            run_stage(["actions", "--cv_repeats", CV_REPEATS] + flag + SETS, f"{sub}.log")
    # results_actions_probe_answer is the fair item 1 test: the kept Llama answer is the one whose hidden state the probe reads
    display(Markdown(open(f"{RUN_DIR}/results_actions_probe_answer/summary.md").read()))
    for f in sorted(glob.glob(f"{RUN_DIR}/results_actions_probe_answer/charts/*.png")):
        display(Image(f))
    if not os.path.exists(f"{RUN_DIR}/results_cpu/summary.md"):
        import importlib.util
        if importlib.util.find_spec("sentence_transformers") is None:
            print("installing sentence-transformers ...")
            !pip install -q sentence-transformers
        run_stage(["cpu", "--cv_repeats", CV_REPEATS] + SETS, "cpu.log")
    display(Markdown(open(f"{RUN_DIR}/results_cpu/summary.md").read()))
    print("Excel workbooks:", f"{RUN_DIR}/results_actions/set6_actions.xlsx", f"{RUN_DIR}/results_cpu/set6_cpu.xlsx")
TIMES["total_session"] = round(time.time() - T_START)
json.dump({"session": SESSION, "run_dir": RUN_DIR, "gpu": gpu, "seconds": TIMES, "versions": ver, "code": EXPECTED_CODE, "complete": ALL_DONE},
          open(f"{SESSION_DIR}/session.json", "w"), indent=1)
print("Wall-clock this session (s):", TIMES)
print("\nALL DONE." if ALL_DONE else "\nNOT COMPLETE YET - run this notebook again to continue.")
try:
    drive.flush_and_unmount()
    print("Saved to Drive.")
except Exception as e:
    print("Drive flush skipped:", e)